In [1]:
%load_ext autoreload
%autoreload 2

import pickle
import ujson
import json
import sys
import os

import pandas as pd
import numpy as np
import torch
import random

from tqdm import tqdm
from collections import defaultdict
from typing import Optional

from bioel.utils.umls_utils import UmlsMappings
from bioel.utils.bigbio_utils import CUIS_TO_REMAP, CUIS_TO_EXCLUDE, DATASET_NAMES, VALIDATION_DOCUMENT_IDS
from bioel.utils.bigbio_utils import load_bigbio_dataset, add_deabbreviations, load_dataset_df, dataset_to_documents, dataset_to_df, load_dataset_df, resolve_abbreviation, dataset_unique_tax_ids
from bioel.utils.solve_abbreviation.solve_abbreviations import create_abbrev

from bioel.ontology import BiomedicalOntology
from bioel.models.arboel.biencoder.data.data_utils import process_ontology
from bioel.evaluate import Evaluate

from torch.utils.data import random_split, DataLoader
from peft import PeftModel
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from sentence_transformers import SentenceTransformer

import argparse
import concurrent
from dotenv import load_dotenv
from tqdm import tqdm
import textgrad as tg
from textgrad.tasks import load_task
from textgrad.autograd.string_based_ops import StringBasedFunction
from textgrad.tasks.big_bench_hard import string_based_equality_fn
import random
load_dotenv(override=True)

import openai
import json
import faiss
from ids import open_ai_api_key
openai.api_key = open_ai_api_key
os.environ["OPENAI_API_KEY"] = open_ai_api_key
import re
import logging
from collections import Counter, defaultdict
from utils_functions import *

/nethome/cye73/conda_envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING 09-17 18:11:47 cuda.py:69] Detected different devices in the system: 
WARNING 09-17 18:11:47 cuda.py:69] Tesla V100-PCIE-32GB
WARNING 09-17 18:11:47 cuda.py:69] NVIDIA A40
WARNING 09-17 18:11:47 cuda.py:69] NVIDIA A40
WARNING 09-17 18:11:47 cuda.py:69] NVIDIA A40
WARNING 09-17 18:11:47 cuda.py:69] Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.


2024-09-17 18:11:48,056	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
INFO:faiss.loader:Loading faiss with AVX512 support.
INFO:faiss.loader:Successfully loaded faiss with AVX512 support.


In [2]:
# !pip install git+https://github.com/ChristopheYe/vllm.git

In [3]:
# Set up logging configuration
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logging.getLogger("httpx").setLevel(logging.WARNING)
sampling_params = SamplingParams(temperature=0, top_p=0.9, max_tokens=20, stop=["<|eot_id|>"])

In [4]:
ontology_dir = "/mitchell/entity-linking/kbs/medic.tsv"
name = "medic"
ontology2 = BiomedicalOntology.load_medic(filepath=ontology_dir, name=name)

# entrez_dict = {"name" : "entrez",
#              "filepath" : "/mitchell/entity-linking/el-robustness-comparison/data/gene_info.tsv",
#              "dataset" : "gnormplus",}
# ontology = BiomedicalOntology.load_entrez(**entrez_dict)


[2024-09-17 18:11:49] [ontology.py] [INFO] Reading entrez from /mitchell/entity-linking/kbs/medic.tsv


In [5]:
dataset_name = 'ncbi_disease'
# dataset_name = 'gnormplus'
path_to_abbrev = "/home2/cye73/data_test2/abbreviations.json"
dataset = load_bigbio_dataset(dataset_name)
dataset = add_deabbreviations(dataset, path_to_abbrev)

In [6]:
dataset_df = dataset_to_df(dataset)
test_df = dataset_df[dataset_df['split'] == 'test']
train_df = dataset_df[dataset_df['split'] == 'train']
# test_df
docs = dataset_to_documents(dataset)

In [7]:
add_full_context(df = test_df, docs=docs)
add_full_context(df = train_df, docs=docs)

_, TestMap_mention2context = add_context(df = test_df, docs = docs)
corpus, TrainMap_mention2context = add_context(df = train_df, docs = docs)
_, _ = add_context(df = dataset_df, docs = docs)
dataset_df

/home2/cye73/llm_disambiguator/experiments/utils_functions.py:209: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["contextualized_mention"] = contextualized_mentions
/home2/cye73/llm_disambiguator/experiments/utils_functions.py:209: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["contextualized_mention"] = contextualized_mentions
/home2/cye73/llm_disambiguator/experiments/utils_functions.py:255: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[ro

,document_id,offsets,text,type,db_ids,split,deabbreviated_text,mention_id,limited_contextualized_mention
2,10021369,"[[43, 76]]",adenomatous polyposis coli tumour,[Modifier],[MESH:D011125],train,adenomatous polyposis coli tumour,10021369.1,"Identification of APC2, a homologue of the [EN..."
3,10021369,"[[93, 132]]",adenomatous polyposis coli (APC) tumour,[Modifier],[MESH:D011125],train,adenomatous polyposis coli (APC) tumour,10021369.2,"Identification of APC2, a homologue of the ade..."
1,10021369,"[[357, 372]]",colon carcinoma,[Modifier],[MESH:D003110],train,colon carcinoma,10021369.3,coli (APC) tumour-suppressor protein controls ...
4,10021369,"[[955, 970]]",colon carcinoma,[Modifier],[MESH:D003110],train,colon carcinoma,10021369.4,"SAMP domains, both of which are required for b..."
0,10021369,"[[1090, 1096]]",cancer,[SpecificDisease],[MESH:D009369],train,cancer,10021369.5,as demonstrated using transient transcriptiona...
...,...,...,...,...,...,...,...,...,...
6880,9988281,"[[996, 1015]]",breast malignancies,[SpecificDisease],[MESH:D001943],test,breast malignancies,9988281.7,"to resolve this issue, we have comprehensively..."
6870,9988281,"[[1123, 1147]]",invasive lobular cancers,[DiseaseClass],[MESH:D018275],test,invasive lobular cancers,9988281.8,localized in discrete nuclear foci in all epit...
6871,9988281,"[[1152, 1179]]",low-grade ductal carcinomas,[SpecificDisease],[MESH:D044584],test,low-grade ductal carcinomas,9988281.9,"foci in all epithelial cell lines, including t..."
6873,9988281,"[[1269, 1286]]",ductal carcinomas,[SpecificDisease],[MESH:D044584],test,ductal carcinomas,9988281.10,of human breast specimens also revealed BRCA1 ...


In [8]:
nlp_model = SentenceTransformer('princeton-nlp/sup-simcse-bert-base-uncased')
# Generate embeddings for the corpus
corpus_embeddings = nlp_model.encode(corpus, convert_to_tensor=True)

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: princeton-nlp/sup-simcse-bert-base-uncased
/nethome/cye73/conda_envs/llm1/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Batches: 100%|██████████| 161/161 [00:09<00:00, 16.29it/s]


In [9]:
corpus_embeddings = corpus_embeddings.cpu().detach().numpy()

# Assuming corpus_embeddings is already a NumPy array
embedding_dimension = corpus_embeddings.shape[1]

# Create the HNSW index with the correct arguments
M = 32  # Number of neighbors in the HNSW graph
index = faiss.IndexHNSWFlat(embedding_dimension, M)

# Normalize the corpus embeddings if using cosine similarity
faiss.normalize_L2(corpus_embeddings)

# Add the embeddings to the index
index.add(corpus_embeddings)

# Print the number of sentences added to the index
print(f"Number of sentences in the index: {index.ntotal}")

Number of sentences in the index: 5134


In [10]:
TrainMap_context2mention = {v: k for k, v in TrainMap_mention2context.items()}

# Load arboel results

In [11]:
dataset_names = ["ncbi_disease"]
model_names = ["arboel_biencoder", "arboel_crossencoder"]
path_to_result = {"ncbi_disease": {
        "arboel_biencoder": "/home2/cye73/results2/arboel/ncbi_disease/biencoder_output_eval.json",
        "arboel_crossencoder": "/home2/cye73/results2/arboel/ncbi_disease/crossencoder_output_eval.json"
    }}

# dataset_names = ["gnormplus"]
# model_names = ["arboel_biencoder", "arboel_crossencoder"]
# path_to_result = {"gnormplus": {
#         "arboel_biencoder": "/home2/cye73/results2/arboel/gnormplus/biencoder_output_eval.json",
#         "arboel_crossencoder": "/home2/cye73/results2/arboel/gnormplus/crossencoder_output_eval.json"
#     }}

abbreviations_path = "/home2/cye73/data_test/abbreviations.json"

evaluator = Evaluate(dataset_names, model_names, path_to_result,
                     abbreviations_path
                     )
evaluator.load_results()
evaluator.process_datasets()
evaluator.evaluate(
    eval_strategies=['basic']
                   )
# evaluator.plot_results(
#     # eval_strategies=['basic']
#     )

dataset : ncbi_disease, model : arboel_biencoder
dataset : ncbi_disease, model : arboel_crossencoder


  0%|          | 0/1 [00:00<?, ?it/s]/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py:249: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda x: min_hit_index(x[0], x[1], eval_mode=eval_mode), axis=1
/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py:249: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda x: min_hit_index(x[0], x[1], eval_mode=eval_mode), axis=1
/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py:249: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with D

Eval Strategy: basic
ncbi_disease


In [12]:
# results = evaluator.full_results["basic"]["gnormplus"]
results = evaluator.full_results["basic"]["ncbi_disease"]
results
cols = ['document_id', 'offsets', 'deabbreviated_text', 'db_ids', 'mention_id', 'joined_offsets', 'arboel_biencoder_resolve_abbrev', 'arboel_biencoder_resolve_abbrev_min_hit_index', 'arboel_crossencoder_resolve_abbrev', 'arboel_crossencoder_resolve_abbrev_min_hit_index']
filtered_results = results[cols].rename(columns={'arboel_biencoder_resolve_abbrev': 'biencoder_candidates',
                                                 'arboel_crossencoder_resolve_abbrev': 'crossencoder_candidates',
                                                 'arboel_biencoder_resolve_abbrev_min_hit_index': 'biencoder_hit_index',
                                                 'arboel_crossencoder_resolve_abbrev_min_hit_index': 'crossencoder_hit_index'})
filtered_results = filtered_results[filtered_results['biencoder_hit_index'] < 10]


In [13]:
biencoder_res = 0
for idx, row in filtered_results.iterrows():
    if row["biencoder_hit_index"] == 0 :
        biencoder_res += 1

print("biencoder results :", biencoder_res/len(filtered_results))

crossencoder_res = 0
for idx, row in filtered_results.iterrows():
    if row["crossencoder_hit_index"] == 0 :
        crossencoder_res += 1

print("crossencoder results :", crossencoder_res/len(filtered_results))

biencoder results : 0.8970775095298602
crossencoder results : 0.9326556543837357


# Data

In [14]:

train_mentions = []
train_mention2context = {}
train_mention2gold = {}
train_mention2text = {}
for idx, row in train_df.iterrows():
    train_mention2gold[row['mention_id']] = row['db_ids']
    train_mentions.append(row['mention_id'])
    train_mention2text[row['mention_id']] = row['deabbreviated_text']
    train_mention2context[row["mention_id"]] = row["limited_contextualized_mention"]

# train_mention2gold

In [15]:
number_candidates = 20

mention2context = {}
for idx, row in test_df.iterrows():
    mention2context[row["mention_id"]] = row["limited_contextualized_mention"]

mentions = []
mention2biencoder_candidates = {}
mention2crossencoder_candidates = {}
mention2gold = {}
mention2hit = {}
mention2text = {}
for idx, row in filtered_results.iterrows():
    # print(idx)
    # Only consider row if hit_index < max number of candidates
    if row['biencoder_hit_index'] < number_candidates:
        mention2biencoder_candidates[row['mention_id']] = [el[0] for el in row['biencoder_candidates'][:number_candidates]]
        # mention2crossencoder_candidates[row['mention_id']] = [el[0] for el in row['crossencoder_candidates'][:number_candidates]]
        mention2gold[row['mention_id']] = row['db_ids']
        mentions.append(row['mention_id'])
        mention2text[row['mention_id']] = row['deabbreviated_text']
        mention2hit[row['mention_id']] = row['biencoder_hit_index']
print(len(mention2hit))
# mention2hit
# len(test_df)

787


# TextGrad

### Data

In [16]:
import torch
from torch.utils.data import Dataset, DataLoader

class EntityDataset(Dataset):
    def __init__(self, 
                 mention2text, 
                 mention2context, 
                 mention2biencoder_candidates, 
                 mention2gold, 
                 index, 
                 ontology, 
                 TrainMap_context2mention, 
                 train_mention2text, 
                 train_mention2gold,
                 corpus):
        self.mention_ids = list(mention2text.keys())
        self.mention2text = mention2text
        self.mention2context = mention2context
        self.mention2biencoder_candidates = mention2biencoder_candidates
        self.mention2gold = mention2gold
        self.index = index
        self.ontology = ontology
        self.TrainMap_context2mention = TrainMap_context2mention
        self.train_mention2text = train_mention2text
        self.train_mention2gold = train_mention2gold
        self.corpus = corpus
    
    def __len__(self):
        return len(self.mention_ids)
    
    def __getitem__(self, idx):
        mention_id = self.mention_ids[idx]
        
        if mention_id not in self.mention2text or mention_id not in self.mention2context \
            or mention_id not in self.mention2biencoder_candidates or mention_id not in self.mention2gold:
            raise KeyError(f"Missing key: {mention_id}")
        
        mention = self.mention2text[mention_id]
        context = self.mention2context[mention_id]
        candidates = get_candidates_data(self.mention2biencoder_candidates[mention_id], ontology2)
        gold_cuis = self.mention2gold[mention_id]
        topk = topk_examples(model = nlp_model,
                    index = self.index,
                    query=context, 
                    corpus=self.corpus, 
                    TrainMap_context2mention=self.TrainMap_context2mention, 
                    train_mention2text=self.train_mention2text, 
                    train_mention2gold=self.train_mention2gold, 
                    ontology=self.ontology, 
                    k=3)
        
        x = {
            'examples': topk,
            'mention': mention,
            'context': context,
            'candidates': candidates  # List of dictionaries with entity info
        }
        y = gold_cuis[0]  # CUIs corresponding to the mention
        z = mention_id
        return x, y, z

In [17]:
dataset = EntityDataset(mention2text=mention2text, 
                        mention2context=mention2context, 
                        mention2biencoder_candidates=mention2biencoder_candidates, 
                        mention2gold=mention2gold,
                        index=index,
                        ontology=ontology2,
                        TrainMap_context2mention=TrainMap_context2mention,
                        train_mention2text=train_mention2text,
                        train_mention2gold=train_mention2gold,
                        corpus=corpus)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)


In [18]:
def flatten_x(x_dict):
    """
    Flattens the dictionary 'x' by combining its 'examples', 'mention', 'context',
    and 'candidates' into a readable string.
    """
    examples = x_dict.get('examples', '')
    mention = x_dict.get('mention', '')
    context = x_dict.get('context', '')

    # For candidates, extract CUIs and names into a readable format
    candidates = x_dict.get('candidates', {})
    candidate_strs = []
    for cui, candidate_data in candidates.items():
        candidate_name = candidate_data.get('name', 'Unknown')
        candidate_strs.append(f"{cui}: {candidate_name}")
    
    candidates_str = "; ".join(candidate_strs)
    
    # Combine all components into a single string
    flattened_str = f"Examples: {examples}\nMention: {mention}\nContext: {context}\nCandidates: {candidates_str}"
    return flattened_str

In [19]:
# Custom collate function because there was a key error
def custom_collate_fn(batch):
    x_batch = []
    y_batch = []
    z_batch = []
    
    for x, y, z in batch:
        x_str = flatten_x(x)
        x_batch.append(x_str)
        y_batch.append(y)
        z_batch.append(z)
    
    return x_batch, y_batch, z_batch

train_size = int(0.01 * len(dataset))
val_size = int(0.01 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

# Separate DataLoaders for each split
batch_size = 2
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=custom_collate_fn)

In [20]:
len(train_loader), len(val_loader), len(test_loader)

(4, 4, 387)

In [21]:
# Iterate over the DataLoader
for i, (x, y, z) in enumerate(train_loader):
    print(i)
    print(x)
    print(y)
    print(z)
    break

Batches: 100%|██████████| 1/1 [00:00<00:00, 46.99it/s]


(1, 768)


Batches: 100%|██████████| 1/1 [00:00<00:00, 130.89it/s]

(1, 768)
0
['Examples: Mention 1: Duchenne muscular dystrophy || Context: between the mildly affected mother (160 repeats; normal 27 repeats) and her more severely affected son (650 repeats), and his sister (650 repeats). The propositus was an isolated case of [ENTITY_START] Duchenne muscular dystrophy [ENTITY_END] with marked dystrophin deficiency in muscle biopsy. The patient was still ambulatory post age 16. Myotonic dystrophy could interfere to some extent with the progression of Duchenne dystrophy. However, other || Correct CUI: {\'MESH:D020388\': {\'cui\': \'MESH:D020388\', \'name\': \'Muscular Dystrophy, Duchenne\', \'types\': \'Disease\', \'aliases\': "Becker Muscular Dystrophy|Becker\'s Muscular Dystrophy|BMD|Cardiomyopathy, Dilated, 3B|Cardiomyopathy, Dilated, X-Linked|Childhood Muscular Dystrophy, Pseudohypertrophic|Childhood Pseudohypertrophic Muscular Dystrophy|DMD|Duchenne and Becker Muscular Dystrophy|Duchenne Becker Muscular Dystrophy|Duchenne-Becker Muscular Dystrophy|

## Model

In [22]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)

set_seed(12)

In [23]:
# Set CUDA devices to see both GPUs
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
# device_1 = torch.device("cuda:0")
# device_2 = torch.device("cuda:1")

with torch.cuda.device(0):
    llm_api_test = tg.get_engine(engine_name="vllm-meta-llama/Meta-Llama-3.1-8B-Instruct", 
                                dtype='half', 
                                enforce_eager=True, 
                                gpu_memory_utilization=0.5,
                                max_model_len = 20000,
                                tensor_parallel_size=1,
                                device = "cuda:0")
                             
# with torch.cuda.device(1):
#     llm_api_eval = tg.get_engine(engine_name="vllm-mistralai/Mistral-7B-Instruct-v0.3", 
#                                 dtype='half', 
#                                 enforce_eager=True, 
#                                 gpu_memory_utilization=0.5,
#                                 max_model_len = 20000,
#                                 tensor_parallel_size=1,
#                                 device = "cuda:1" )


# llm_api_eval = tg.get_engine(engine_name="gpt-4o-2024-08-06")
llm_api_eval = tg.get_engine(engine_name="gpt-4o-mini")

tg.set_backward_engine(llm_api_eval, override=True)

WARNING 09-17 18:12:08 config.py:1657] Casting torch.bfloat16 to torch.float16.
WARNING 09-17 18:12:08 config.py:370] Async output processing is only supported for CUDA or TPU. Disabling it for other platforms.
INFO 09-17 18:12:08 llm_engine.py:223] Initializing an LLM engine (v0.6.1.post2) with config: model='meta-llama/Meta-Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=20000, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda:0, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(ot

/nethome/cye73/conda_envs/llm1/lib/python3.9/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/nethome/cye73/conda_envs/llm1/lib/python3.9/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


INFO 09-17 18:12:10 model_runner.py:1077] Starting to load model meta-llama/Meta-Llama-3.1-8B-Instruct...
INFO 09-17 18:12:10 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 09-17 18:12:10 selector.py:116] Using XFormers backend.
INFO 09-17 18:12:10 weight_utils.py:242] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:01<00:05,  1.97s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:02<00:02,  1.17s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:04<00:01,  1.66s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:06<00:00,  1.81s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:06<00:00,  1.72s/it]



INFO 09-17 18:12:18 model_runner.py:1090] Loading model weights took 14.9888 GB
INFO 09-17 18:12:21 gpu_executor.py:122] # GPU blocks: 2429, # CPU blocks: 2048


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [24]:
k = 2
system_instructions = """You are a professional data annotator and curator.
Your task is to identify the correct entity for a given mention based on the provided context and the descriptions of {k} candidate entities."""

In [25]:
# mention = "9585611.10"
# text = mention2text[mention]
# context = mention2context[mention]
# candidates = get_candidates_data(mention2biencoder_candidates[mention], ontology2)
# topk = topk_examples(model = nlp_model, # sentence transformer model
#                     index=index,
#                     query=context,
#                     corpus=corpus,
#                     TrainMap_context2mention=TrainMap_context2mention,
#                     train_mention2text=train_mention2text,
#                     train_mention2gold=train_mention2gold,
#                     ontology=ontology2,
#                     k=k)

# # Generate the prediction using LLaMA 8B
# pred_cui = prompt_vllm(mention=text, 
#                        context=context, 
#                        system_instructions=system_instructions,
#                        candidates=candidates, 
#                        topk_examples=topk,
#                        llm=llm_api_test.client,
#                        tokenizer=llm_api_test.tokenizer,
#                        sampling_params=sampling_params)

# # Output the results
# print("Mention ID:", mention)
# print("Context:", context)
# print("Top k examples:", topk)
# print("Text:", mention2text[mention])
# print("Gold CUI:", mention2gold[mention])
# print("Predicted CUI:", pred_cui)

In [26]:
system_prompt = f"""
You are a professional data annotator and curator.
Your task is to identify the correct entity for a given mention based on the provided context and the descriptions of k candidate entities.

You will be provided with a number of examples to better understand the task.

For each mention that you will have to link, you will be provided with the "mention", the "context" where it appears, and a list of candidate entities "candidates" to choose from 

You MUST PROVIDE an ANSWER among the candidates. \n

Return the answer in the following format: CUI (e.g., "MESH:D000000" or "OMIM:000000" are valid answers).
Use step-by-step reasoning internally, but only provide the final answer without any explanations.
"""

In [27]:
model_prompt = tg.Variable(system_prompt, 
                            requires_grad=True, 
                            role_description="structured system prompt to a somewhat capable language model that specifies the behavior and strategies for entity linking task")
model = tg.BlackboxLLM(llm_api_test, model_prompt)

eval_prompt = tg.Variable(system_prompt, 
                            requires_grad=True,
                            role_description="system prompt to the language model")

model_eval = tg.BlackboxLLM(llm_api_eval, eval_prompt)

optimizer = tg.TextualGradientDescent(engine=llm_api_eval, parameters=[model_prompt])


In [28]:
def cui_equality_fn(prediction: tg.Variable, ground_truth: tg.Variable) -> bool:
    """
    Custom function to compare the CUI of the prediction and ground truth.
    :param prediction: The predicted entity with CUI.
    :param ground_truth: The ground truth entity with CUI.
    :return: True if CUIs match, else False.
    """
    # Extract the string value from the tg.Variable
    predicted_cui = extract_cui(prediction.value)  # Extract CUI from the string stored in prediction.value
    ground_truth_cui = extract_cui(ground_truth.value)  # Extract CUI from the string stored in ground_truth.value
    
    return int(predicted_cui == ground_truth_cui)

def extract_cui(text):
    """
    Extracts the CUI from the text generated by the LLM.
    """
    # Define the regular expression pattern to match "MESH" or "OMIM"
    pattern = r"\b(MESH|OMIM|NCBIGene):[A-Za-z0-9]+\b"

    # Search for the pattern in the text
    match = re.search(pattern, text)

    # Return the matched string if found, otherwise return None
    return match.group(0) if match else None

fn_purpose = "The runtime of string-based function that checks if the prediction is correct."
eval_fn = StringBasedFunction(cui_equality_fn, function_purpose=fn_purpose)

In [29]:
# def eval_sample(item, eval_fn, model):
#     """
#     This function allows us to evaluate if an answer to a question in the prompt is a good answer.

#     """
#     x, y, z = item
#     print(f"Before conversion - x: {x}, type: {type(x)}")
#     x = tg.Variable(str(x), requires_grad=False, role_description="query to the language model")
#     y = tg.Variable(y, requires_grad=False, role_description="correct answer for the query")
    
#     print(f"After conversion - x: {x.value}, type: {type(x.value)}")
#     print(f"After conversion - y: {y.value}, type: {type(y.value)}")
#     response = model(x)
#     try:
#         eval_output_variable = eval_fn(inputs=dict(prediction=response, ground_truth_answer=y))
#         return int(eval_output_variable.value)
#     except:
#         eval_output_variable = eval_fn([x, y, response])
#         eval_output_parsed = eval_fn.parse_output(eval_output_variable)
#         return int(eval_output_parsed)
    
def eval_sample(item, eval_fn, model):
    """
    This function allows us to evaluate if an answer to a question in the prompt is a good answer.
    This now handles both single items and batches of items.
    """
    if isinstance(item, list):
        print("ICCCCIIIIII")
        # If item is a list (i.e., a batch), process each element
        results = []
        for single_item in item:
            x, y, z = single_item
            print(f"Before conversion - x: {x}, type: {type(x)}")
            x = tg.Variable(x, requires_grad=False, role_description="query to the language model")
            y = tg.Variable(y, requires_grad=False, role_description="correct answer for the query")
            print(f"After conversion - x: {x.value}, type: {type(x.value)}")
            print(f"After conversion - y: {y.value}, type: {type(y.value)}")
            response = model(x)
            eval_output_variable = eval_fn(inputs=dict(prediction=response, ground_truth_answer=y))
            results.append(int(eval_output_variable.value))
        return results
    else:
        # Single item case
        x, y, z = item
        print(f"Before conversion - x: {x}, type: {type(x)}")
        x = tg.Variable(x, requires_grad=False, role_description="query to the language model")
        y = tg.Variable(y, requires_grad=False, role_description="correct answer for the query")
        print(f"After conversion - x: {x.value}, type: {type(x.value)}")
        print(f"After conversion - y: {y.value}, type: {type(y.value)}")
        response = model(x)
        eval_output_variable = eval_fn(inputs=dict(prediction=response, ground_truth_answer=y))
        return int(eval_output_variable.value)
    
def eval_dataset(test_set, eval_fn, model, max_samples: int=None):
    if max_samples is None:
        max_samples = len(test_set)
    accuracy_list = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        futures = []
        for _, sample in enumerate(test_set):
            
            future = executor.submit(eval_sample, sample, eval_fn, model)
            futures.append(future)
            if len(futures) >= max_samples:
                break
        tqdm_loader = tqdm(concurrent.futures.as_completed(futures), total=len(futures), position=0)
        for future in tqdm_loader:
            acc_item = future.result()
            accuracy_list.append(acc_item)
            tqdm_loader.set_description(f"Accuracy: {np.mean(accuracy_list)}")
    return accuracy_list 

def run_validation_revert(system_prompt: tg.Variable, results, model, eval_fn, val_set):
    val_performance = np.mean(eval_dataset(val_set, eval_fn, model))
    previous_performance = np.mean(results["validation_acc"][-1])
    print("val_performance: ", val_performance)
    print("previous_performance: ", previous_performance)
    previous_prompt = results["prompt"][-1]
    
    if val_performance < previous_performance:
        print(f"rejected prompt: {system_prompt.value}")
        system_prompt.set_value(previous_prompt)
        val_performance = previous_performance

    results["validation_acc"].append(val_performance)

In [30]:
n_samples = len(train_loader) * batch_size
for epoch in range(2):
    for steps, (batch_x, batch_y, batch_z) in enumerate((pbar := tqdm(train_loader, position=0))):
        pbar.set_description(f"Training step {steps}. Epoch {epoch}")
        optimizer.zero_grad()
        losses = []
        
        for (x, y, z) in zip(batch_x, batch_y, batch_z):
            # Convert each dynamic part into a tg.Variable
            x = tg.Variable(x, requires_grad=False, role_description="query to analyze for entity linking")
            # Convert target answer into a Variable
            y = tg.Variable(y, requires_grad=False, role_description="correct answer for the query")
            
            # Response from the model using the generated complete prompt
            response = model(x)

            # Now pass the tg.Variable to the model_test function
            print('mention ID:', z)
            print("response :", response, type(response))
            print("y :", y, type(y))
            eval_output_variable = eval_fn(inputs=dict(prediction=response, ground_truth=y))
            losses.append(eval_output_variable)
        
        # Sum and backpropagate loss
        total_loss = tg.sum(losses)

        total_loss.backward()
        optimizer.step()
        
        run_validation_revert(system_prompt=model_prompt, results=results, model=model_eval, eval_fn=eval_fn, val_set=val_loader)
        print("Optimized system prompt: ", model_prompt.get_value())

        results["prompt"].append(model_prompt.get_value())
        
        if steps == 3:
            break


Batches: 100%|██████████| 1/1 [00:00<00:00, 83.85it/s]


(1, 768)


Training step 0. Epoch 0:   0%|          | 0/4 [00:00<?, ?it/s]

(1, 768)


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.13it/s, est. speed input: 2123.70 toks/s, output: 11.37 toks/s]
INFO:textgrad:LLMCall function forward
INFO:textgrad:StringBasedFunction


...

(1, 768)
Before conversion - x: ['Examples: Mention 1: hemochromatosis || Context: [ENTITY_START] hemochromatosis (HFE) gene on chromosome 6, elevated serum transferrin saturation, and excess iron deposits throughout the body. To assess the prevalence and clinical expression of the HFE gene, we conducted a population-based study in Busselton, Australia. In 1994, we obtained blood samples for the determination of serum transferrin saturation and ferritin levels and the presence or absence of the C282Y mutation and the H63D mutation (which may contribute to increased hepatic iron levels) in 3011 unrelated white adults. We evaluated all subjects who had persistently elevated transferrin-saturation values (45 percent or higher) or were homozygous for the C282Y mutation. We recommended liver biopsy for subjects with serum ferritin levels of 300 ng per milliliter or higher. The subjects were followed for up to four years. RESULTS Sixteen of the subjects (0. 5 percent) were homozygous for the

Batches: 100%|██████████| 1/1 [00:00<00:00, 135.13it/s]


(1, 768)
Before conversion - x: ["Examples: Mention 1: X-linked agammaglobulinemia || Context: residue which has been involved in InsP binding in PH domains of other proteins. The same residue is often mutated in the Brutons tyrosine kinase (Btk) gene in patients with an [ENTITY_START] X-linked agammaglobulinemia. [ENTITY_END] The Arg610Gln mutation represents the first case of a mutation in the PH domain of the FGD1 gene and additional evidence that mutations in PH domains can be associated to human || Correct CUI: {'OMIM:300755': {'error': 'Entity for OMIM:300755 not found'}}\nMention 2: optic atrophy || Context: molecular genetic analysis of 19 Wolfram syndrome kindreds demonstrating a wide spectrum of mutations in WFS1. Wolfram syndrome is an autosomal recessive neurodegenerative disorder characterized by juvenile-onset diabetes mellitus and progressive [ENTITY_START] optic atrophy. [ENTITY_END] mtDNA deletions have been described, and a gene (WFS1) recently has been identified, on

Training step 0. Epoch 0:   0%|          | 0/4 [00:19<?, ?it/s]


AssertionError: Value must be a string, int, or image (bytes). Got: <class 'list'>

# THEIR EXAMPLE

In [67]:
def eval_sample(item, eval_fn, model):
    """
    This function allows us to evaluate if an answer to a question in the prompt is a good answer.

    """
    x, y = item
    x = tg.Variable(x, requires_grad=False, role_description="query to the language model")
    y = tg.Variable(y, requires_grad=False, role_description="correct answer for the query")
    response = model(x)
    try:
        eval_output_variable = eval_fn(inputs=dict(prediction=response, ground_truth_answer=y))
        return int(eval_output_variable.value)
    except:
        eval_output_variable = eval_fn([x, y, response])
        eval_output_parsed = eval_fn.parse_output(eval_output_variable)
        return int(eval_output_parsed)
    
def eval_dataset(test_set, eval_fn, model, max_samples: int=None):
    if max_samples is None:
        max_samples = len(test_set)
    accuracy_list = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        futures = []
        for _, sample in enumerate(test_set):
            
            future = executor.submit(eval_sample, sample, eval_fn, model)
            futures.append(future)
            if len(futures) >= max_samples:
                break
        tqdm_loader = tqdm(concurrent.futures.as_completed(futures), total=len(futures), position=0)
        for future in tqdm_loader:
            acc_item = future.result()
            accuracy_list.append(acc_item)
            tqdm_loader.set_description(f"Accuracy: {np.mean(accuracy_list)}")
    return accuracy_list 

def run_validation_revert(system_prompt: tg.Variable, results, model, eval_fn, val_set):
    val_performance = np.mean(eval_dataset(val_set, eval_fn, model))
    previous_performance = np.mean(results["validation_acc"][-1])
    print("val_performance: ", val_performance)
    print("previous_performance: ", previous_performance)
    previous_prompt = results["prompt"][-1]
    
    if val_performance < previous_performance:
        print(f"rejected prompt: {system_prompt.value}")
        system_prompt.set_value(previous_prompt)
        val_performance = previous_performance

    results["validation_acc"].append(val_performance)

In [64]:
set_seed(12)
# llm_api_eval = tg.get_engine(engine_name="gpt-4o-2024-08-06")
# llm_api_test = tg.get_engine(engine_name="gpt-4o-mini")
tg.set_backward_engine(llm_api_eval, override=True)
# Load the data and the evaluation function
train_set, val_set, test_set, eval_fn = load_task("BBH_object_counting", evaluation_api=llm_api_eval)
print("Train/Val/Test Set Lengths: ", len(train_set), len(val_set), len(test_set))
STARTING_SYSTEM_PROMPT = train_set.get_task_description()


Train/Val/Test Set Lengths:  50 100 100


In [65]:
# Convert the 'y' column to a list of Python integers
y_as_python_int = [int(value) for value in test_set.data['y']]

# Reassign this list back to the 'y' column as an object dtype to avoid numpy conversion
test_set.data['y'] = pd.Series(y_as_python_int, dtype="object")

# Verify the type of the elements
print(type(test_set.data.iloc[0]['y']))  # This should return <class 'int'>

<class 'int'>


In [ ]:
train_loader = tg.tasks.DataLoader(train_set, batch_size=3, shuffle=True)


# Testing the 0-shot performance of the evaluation engine
system_prompt = tg.Variable(STARTING_SYSTEM_PROMPT, 
                            requires_grad=True, 
                            role_description="system prompt to the language model")
model_evaluation = tg.BlackboxLLM(llm_api_eval, system_prompt)

system_prompt = tg.Variable(STARTING_SYSTEM_PROMPT, 
                            requires_grad=True,
                            role_description="structured system prompt to a somewhat capable language model that specifies the behavior and strategies for the QA task")
model = tg.BlackboxLLM(llm_api_test, system_prompt)

optimizer = tg.TextualGradientDescent(engine=llm_api_eval, parameters=[system_prompt])

results = {"test_acc": [], "prompt": [], "validation_acc": []}
results["test_acc"].append(eval_dataset(test_set, eval_fn, model))
results["validation_acc"].append(eval_dataset(val_set, eval_fn, model))
results["prompt"].append(system_prompt.get_value())


In [ ]:
for epoch in range(3):
    for steps, (batch_x, batch_y) in enumerate((pbar := tqdm(train_loader, position=0))):
        print(len(batch_x))
        pbar.set_description(f"Training step {steps}. Epoch {epoch}")
        optimizer.zero_grad()
        losses = []
        for (x, y) in zip(batch_x, batch_y):
            x = tg.Variable(x, requires_grad=False, role_description="query to the language model")
            y = tg.Variable(y, requires_grad=False, role_description="correct answer for the query")
            response = model(x)
            try:
                eval_output_variable = eval_fn(inputs=dict(prediction=response, ground_truth_answer=y))
            except:
                eval_output_variable = eval_fn([x, y, response])
            losses.append(eval_output_variable)
        total_loss = tg.sum(losses)
        total_loss.backward()
        optimizer.step()
        
        run_validation_revert(system_prompt, results, model, eval_fn, val_set)
        
        print("sys prompt: ", system_prompt)
        test_acc = eval_dataset(test_set, eval_fn, model)
        results["test_acc"].append(test_acc)
        results["prompt"].append(system_prompt.get_value())
        if steps == 3:
            break

In [104]:
type(test_set.data.iloc[0]['y'])

numpy.int64